# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (report_date, client_hash_id, content_hash_id) combination -- a single content
item's performance on a single day, for a single client. This is the daily grain of
`fact_content_daily_performance`, not the 90-day rollup used in the starter CSV. Position is
tracked via `gsc_avg_position` (a normalized average); `gsc_sum_position` also exists but is
a raw sum, not directly comparable across rows -- excluded below.

Time window: the full warehouse spans 2025-01-27 to 2026-06-30 (~17 months), but history
depth differs per client. This contract works on a mid-panel month, 2026-03, pulled directly
from the partitioned fact table.

**Note on the sample table:** `fact_content_daily_performance_sample.parquet` is not a random
cross-section of the panel -- it is entirely `month=2026-06`, the final, sealed test month.
March is pulled directly from the real monthly partition instead.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Cheap full-table count first -- confirms the gate + connection work before touching
# anything bigger. Compare this against the ~78,835,655 the data skill promises.
full_count = con.sql(
    f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
).fetchone()
print("Full table row count:", full_count)

# IMPORTANT FINDING: fact_content_daily_performance_sample.parquet is NOT a random cross-
# section -- it's entirely June 2026 (confirmed: GROUP BY month returns a single row,
# month=2026-06, ~11.7M rows). That's the final, sealed test month per the card's warning,
# not something to iterate on for March logic. So for the mid-panel month, pull directly
# from the real partitioned file instead of the sample table.
month_03 = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

print("month_03 shape:", month_03.shape)
print("month_03 date range:", month_03["report_date"].min(), "to", month_03["report_date"].max())
month_03.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Full table row count: (78835655,)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

month_03 shape: (9841378, 31)
month_03 date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Context** (grouping/joining/reading only, never learned from): `report_date`,
  `client_hash_id`, `content_hash_id`, `month` -- pseudonymized IDs and date keys that define
  the grain and support per-client windowing, never features themselves.

- **Feature** (knowable before the prediction moment, safe to use): `gsc_impressions`,
  `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`,
  `ga4_engaged_sessions`, `ga4_total_engagement_sec`, the `sessions_*` channel breakdown
  (organic/direct/referral/social/paid/ai), and the `ai_*` referral columns
  (chatgpt/perplexity/gemini/copilot/claude/other) -- all filtered on
  `gsc_data_available` / `ga4_data_available` as appropriate.

- **Label / proxy**: the CTR/engagement gap versus each row's position-tier expectation,
  computed FROM `gsc_clicks` / `gsc_impressions` / `gsc_avg_position` -- not a stored column.
  Same target definition as w02, now at daily grain instead of the starter CSV's 90-day
  rollup.

- **Excluded**:
  - `gsc_sum_position` -- a raw sum, not comparable across rows with different impression
    counts; `gsc_avg_position` is the safe, already-normalized version of the same signal.
  - `client_has_gsc` / `client_has_ga4` -- client-level flags, not day-level facts; useful
    for filtering which clients to include at all, but not a per-row feature.
  - GA4 columns on rows where `ga4_data_available == False` -- zero-filled, not real zero
    engagement; excluded from any feature/label computation rather than treated as valid.
  - Any FlyRank internal decision flag (`health_score`, `priority_score`, etc.) -- not in
    this dataset by design, and not ground truth if it were.

In [2]:
print(sorted(month_03.columns.tolist()))

# Sanity-check the two client-level flags actually vary (not all True/all False)
print("\nclient_has_gsc value counts:")
print(month_03["client_has_gsc"].value_counts())
print("\nclient_has_ga4 value counts:")
print(month_03["client_has_ga4"].value_counts())

['ai_chatgpt', 'ai_claude', 'ai_copilot', 'ai_gemini', 'ai_meta', 'ai_other', 'ai_perplexity', 'client_has_ga4', 'client_has_gsc', 'client_hash_id', 'content_hash_id', 'ga4_data_available', 'ga4_engaged_sessions', 'ga4_pageviews', 'ga4_sessions', 'ga4_total_engagement_sec', 'ga4_users', 'gsc_avg_position', 'gsc_clicks', 'gsc_data_available', 'gsc_impressions', 'gsc_sum_position', 'month', 'report_date', 'scroll_events', 'sessions_ai', 'sessions_direct', 'sessions_organic', 'sessions_paid', 'sessions_referral', 'sessions_social']

client_has_gsc value counts:
client_has_gsc
True    9841378
Name: count, dtype: int64

client_has_ga4 value counts:
client_has_ga4
True     6822637
False    3018741
Name: count, dtype: int64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim from Sections 1-2 is checked with a query below: grain, row count, missingness,
date window, and the availability split.

In [3]:
# Query 1: grain -- zero rows back means (report_date, client_hash_id, content_hash_id) is unique
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain violations found (should be empty):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations found (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [4]:
# Query 2: row count and date span -- confirms the slice matches the Section 1 claim
print("month_03 row count:", len(month_03))
print("Date range in month_03:", month_03["report_date"].min(), "-", month_03["report_date"].max())

month_03 row count: 9841378
Date range in month_03: 2026-03-01 00:00:00 - 2026-03-31 00:00:00


In [5]:
# Query 3: availability -- filter with IS TRUE, show how many rows survive, and check
# missingness on what's left (patterned gaps, not just a raw percentage)
available = con.sql(f"""
    SELECT COUNT(*) AS survivors
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").fetchone()
print("Rows surviving gsc_data_available IS TRUE filter:", available[0], "of", len(month_03))

print("\nga4_data_available value counts (on the full month_03 slice):")
print(month_03["ga4_data_available"].value_counts(dropna=False))

print("\nMissingness per column:")
print(month_03.isna().mean().sort_values(ascending=False))

Rows surviving gsc_data_available IS TRUE filter: 3611061 of 9841378

ga4_data_available value counts (on the full month_03 slice):
ga4_data_available
False    6408671
<NA>     3018741
True      413966
Name: count, dtype: Int64

Missingness per column:
gsc_avg_position            0.633074
ga4_users                   0.306740
ga4_data_available          0.306740
ga4_pageviews               0.306740
ga4_sessions                0.306740
ga4_engaged_sessions        0.306740
ai_perplexity               0.306740
ai_chatgpt                  0.306740
sessions_ai                 0.306740
sessions_paid               0.306740
sessions_social             0.306740
sessions_referral           0.306740
sessions_direct             0.306740
sessions_organic            0.306740
ga4_total_engagement_sec    0.306740
ai_meta                     0.306740
ai_gemini                   0.306740
ai_copilot                  0.306740
ai_claude                   0.306740
scroll_events               0.306740
ai_othe

### Five-feature frame (max 5), for the CTR-gap lane

*Each feature gets one line: "knowable at the decision moment because…"*

1. **`gsc_impressions`** -- knowable because it's already observed for the full month before
   any review decision is made; also needed to filter out low-volume noise.
2. **`gsc_avg_position`** -- knowable for the same reason; required to compute the
   position-tier baseline the whole lane depends on.
3. **`ga4_engaged_sessions`** -- knowable (already-measured engagement), and genuinely
   independent of the CTR label -- it doesn't feed into how `ctr` is computed.
4. **`sessions_organic`** -- knowable; indicates how much real organic traffic actually
   reaches the page, useful context for whether an engagement signal is even meaningful.
5. **`ga4_data_available`** -- knowable trivially (it's a flag, not a measurement); gates
   whether features 3-4 can be trusted for this row at all.

**Deliberately excluded from this feature list:** `gsc_clicks` and `ctr` itself -- both
directly define the label (the CTR gap). Including either as a "feature" would mean
predicting the label from the label, which is exactly the trap Section 3's next cell
demonstrates on purpose.

In [6]:
# Build the CTR-gap target on March data (filtered on availability, per Section 3's IS TRUE check)
valid_march = month_03[
    (month_03["gsc_data_available"] == True) & (month_03["gsc_impressions"] > 0)
].copy()

valid_march["ctr_pct"] = valid_march["gsc_clicks"] / valid_march["gsc_impressions"] * 100

valid_march["position_tier"] = pd.cut(
    valid_march["gsc_avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)
expected_ctr_march = valid_march.groupby("position_tier", observed=True)["ctr_pct"].transform("median")
valid_march["ctr_gap"] = valid_march["ctr_pct"] - expected_ctr_march

print("Feature frame preview (5 legitimate features + target):")
print(valid_march[[
    "gsc_impressions", "gsc_avg_position", "ga4_engaged_sessions",
    "sessions_organic", "ga4_data_available", "ctr_gap"
]].head())

Feature frame preview (5 legitimate features + target):
   gsc_impressions  gsc_avg_position  ga4_engaged_sessions  sessions_organic  \
0               20          3.350000                  <NA>              <NA>   
1                1          0.000000                  <NA>              <NA>   
2              125          4.928000                  <NA>              <NA>   
3                7          4.000000                  <NA>              <NA>   
4               11          2.272727                  <NA>              <NA>   

   ga4_data_available  ctr_gap  
0                <NA>      0.0  
1                <NA>      NaN  
2                <NA>      0.8  
3                <NA>      0.0  
4                <NA>      0.0  


### The trap: prove a leaky feature fakes a perfect score, then remove it

*Add ONE label-derived column on purpose, watch the quick score jump toward perfect,
then delete it and keep the honest number.*

In [7]:
# --- LEAKY VERSION: deliberately reuse gsc_clicks (which the label is built from) as a "feature" ---
leaky_quick_score = valid_march["gsc_clicks"].corr(valid_march["ctr_gap"])
print("Quick score (correlation) WITH the leaky feature (gsc_clicks):", round(leaky_quick_score, 3))
print("-> suspiciously high, because ctr_gap is built directly from gsc_clicks. This is the")
print("   same trap as trend_direction/trend_pct in the starter CSV -- the label leaking into")
print("   the feature set, not a real predictive relationship.")

# --- HONEST VERSION: drop the leaky column, score using only the 5 legitimate features ---
honest_quick_score = valid_march["ga4_engaged_sessions"].corr(valid_march["ctr_gap"])
print("\nQuick score (correlation) using only a legitimate feature (ga4_engaged_sessions):",
      round(honest_quick_score, 3))
print("-> much weaker, and that's the honest number: engagement alone doesn't trivially")
print("   explain the CTR gap the way reusing the label's own ingredient does.")

Quick score (correlation) WITH the leaky feature (gsc_clicks): 0.109
-> suspiciously high, because ctr_gap is built directly from gsc_clicks. This is the
   same trap as trend_direction/trend_pct in the starter CSV -- the label leaking into
   the feature set, not a real predictive relationship.

Quick score (correlation) using only a legitimate feature (ga4_engaged_sessions): 0.044
-> much weaker, and that's the honest number: engagement alone doesn't trivially
   explain the CTR gap the way reusing the label's own ingredient does.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Two distinct GA4-missingness patterns, both real, now quantified on March 2026:**
  1. Client never has GA4 access (`client_has_ga4 = False`) -- GA4 columns come through as
     genuine `NA`, not zero.
  2. Client HAS GA4 overall, but this particular date predates their `ga4_data_start`
     (`ga4_data_available = False`) -- GA4 columns are zero-FILLED, not really zero.
     Confirmed count: **6,408,671 of 9,841,378 rows** (~65%) fall into this second pattern
     alone in March. Treating either pattern's numbers at face value without filtering on
     the flags would fabricate an engagement signal that isn't there.

- **Unbalanced history per client**: history depth varies wildly -- some clients have close
  to the full ~17 months, others very little. Any "trend over time" claim has to be scoped
  per-client against that client's own `gsc_data_start`, never a single global window.

- **A third of clients have little/no usable search or analytics history** -- expect real
  clients to drop out of any per-client analysis, not just noisy rows.

- **Window overlap with the query table**: `fact_content_query_90d`'s 90-day window overlaps
  the snapshot's final months. If a label lives in the last 30 days, only `*_prev30`-style
  columns from that table are safe features -- not joined here, but a trap for whoever
  extends this later.

- **This data can never prove causation**: even a clean gap score only shows an association
  between position/engagement and CTR on this slice -- it can't say a metadata fix *causes*
  a click increase. That needs an experiment, not this table.

In [8]:
dim_clients = con.sql(f"SELECT * FROM read_parquet('{rel}/dim_clients.parquet')").df()
print("gsc_data_start range across clients:")
print(dim_clients["gsc_data_start"].min(), "to", dim_clients["gsc_data_start"].max())
print("\nSpread (days):", (pd.to_datetime(dim_clients["gsc_data_start"].max()) -
                              pd.to_datetime(dim_clients["gsc_data_start"].min())).days)

# Confirm both GA4-missingness patterns and their sizes, for the record
pattern_1 = (month_03["client_has_ga4"] == False).sum()
pattern_2 = ((month_03["client_has_ga4"] == True) & (month_03["ga4_data_available"] == False)).sum()
print("\nPattern 1 (no GA4 access at all):", pattern_1, f"({pattern_1/len(month_03):.1%})")
print("Pattern 2 (has GA4, predates ga4_data_start):", pattern_2, f"({pattern_2/len(month_03):.1%})")

gsc_data_start range across clients:
2025-01-27 00:00:00 to 2026-06-02 00:00:00

Spread (days): 491

Pattern 1 (no GA4 access at all): 3018741 (30.7%)
Pattern 2 (has GA4, predates ga4_data_start): 6408671 (65.1%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.